In [12]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings
from pymilvus import MilvusClient
from app.core.config import settings
from pymilvus import Function, FunctionType, DataType
# RRF 混合检索
from pymilvus import AnnSearchRequest
# 权重融合策略
from pymilvus import WeightedRanker
# 默认的RRF融合策略
from pymilvus import AnnSearchRequest, RRFRanker
# cross-encoder rerank 重排
from pymilvus import Function, FunctionType

# 1. 初始化向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)
# 2. 初始化客户端
client = MilvusClient(uri=settings.rag.milvus_url)
#   Collection名称
collection_name = "my_collection_1"

#准备docs文档
def prepare_docs():
    # =============1.准备文档================
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

#创建Collection
def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加BM25函数，可以自动把原始文档转BM25向量存储
    # https://milvus.io/docs/zh/bm25-function.md
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)

    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )

create_collection()

# 查看Collection
res = client.list_collections()

print(res)  # ['my_collection_1']

# 删除Collection
#client.drop_collection(collection_name=collection_name)

# 把文档向量化，要传入list[str]
docs = prepare_docs()
vectors = embeddings.embed_documents([doc.page_content for doc in docs])
# milvus接收的是dict，所以先做数据转换
data = [
    {
        "id": i+1,
        "content": doc.page_content,
        "h2": doc.metadata.get("h2", ""),
        "dense": vectors[i],  # 稠密向量字段
    }
    for i, doc in enumerate(docs)
]
# 插入
res = client.insert(collection_name, data)
print(res)
#{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}
#更新
# 1.准备两条数据

# id为1的数据，修改其h2字段值，模拟更新
data1 = data[0]
data1['h2'] = "测试一下"

 # id为2的数据，将其id修改为88，模拟新增, 如果id不存在，则新增
data2 = data[1]
data2['id'] = 88

# 2.测试upsert
res = client.upsert(collection_name, [data1, data2])
print(res)# {'upsert_count': 2, 'ids': [1, 88]}

#根据id删除
# 根据id删除
res = client.delete(collection_name, ids=[88])
print(res) # {'delete_count': 1, 'cost': 0}

# 定义一个稠密检索的函数
def dense_search(query: str):
    # 1.先把问题向量化
    query_vector = embeddings.embed_query(query)

    # 2.然后搜索：类似:select content,h2 from collection_name where dense ann query_vector 使用余弦值比较 limit 3
    res = client.search(
        collection_name=collection_name,    # Collection名称
        anns_field="dense",                 # 检索字段名（稠密）
        data=[query_vector],                # 问题向量（稠密）
        limit=3,                            # 查询文档最大数量
        search_params={"metric_type": "COSINE"},  # 向量相似度算法，必须和索引时一致
        output_fields=["content", "h2"]  # 要返回的字段名，默认只返回id和score
    )
    # 3.遍历结果
    for hits in res:
        for hit in hits:
            print(hit)

# 定义稀疏检索函数
def sparse_search(query: str):
    # 1.直接检索，内置的hm25函数会自动把query转向量
    res = client.search(
        collection_name=collection_name,  # Collection名称
        data=[query],
        anns_field="sparse",  # 检索字段名（稀疏）
        limit=3,  # 查询文档最大数量
        output_fields=["content", "h2"]  # 要返回的字段名
    )
    # 2.打印结果
    for hits in res:
        for hit in hits:
            print(hit)
# RRF 混合检索
def hybrid_search(query: str, ranker):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)
#创建cross encoder重排方法
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",      # ranker名词，唯一即可
        input_field_names=["content"],          # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 单批次中处理的文档最大数量
        },
    )
#沿用之前的混合检索函数，传递create_cross_encoder_ranker
def cross_encoder_hybrid_search(query: str):
    # 初始化一个cross-encoder的reranker
    cross_encoder_ranker = create_cross_encoder_ranker([query])
    # 混合检索
    hybrid_search(query, cross_encoder_ranker)

#带过滤条件的搜索请求，普通字段过滤，再进行向量检索，从而提高查询效率
def hybrid_search_by_conditions(query: str, ranker, filter_query: str = None):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
        filter=filter_query
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
        filter=filter_query
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)


print("=============== 稠 密 检 索 测试一:教育的负面作用是什么===============")
# 测试，教育的负面作用是语义理解，属于专业名词，稠密向量检索效果可以
dense_search("教育的负面作用是什么")
print("=============== 稠 密 检 索 测试二:给教师的建议是谁写的===============")
# 测试，《给教师的建议》是书的名字，属于专业名词,稠密向量检索效果不好
dense_search("给教师的建议是谁写的")
print("=============== 稀 疏 检 索 测试一:教育的负面作用是什么===============")
# 测试模糊的语义搜索，稀疏检索不太擅长
sparse_search("教育的负面作用是什么")
print("=============== 稀 疏 检 索 测试二:给教师的建议是谁写的===============")
# 专有名称查询：稀疏查询效果良好
sparse_search("给教师的建议是谁写的")
print("=============== 混 合 加 权 融 合 检 索 测试一:教育的负面作用是什么(权重0.5 0.5)===============")
# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)
# 测试
hybrid_search("教育的负面作用是什么", ranker)
print("=============== RRF 融 合 检 索 测试二:教育的负面作用是什么(权重 k = 60)===============")
# 定义RRF融合策略
ranker = RRFRanker(k=60)
# 测试
hybrid_search("教育的负面作用是什么", ranker)
print("=============== RRF 融 合 检 索 + 重 排 测试三:教育的负面作用是什么(权重 k = 60)===============")
cross_encoder_hybrid_search("教育的负面作用是什么")
print("=============== RRF 融 合 检 索 （带 过 滤 条 件 的 搜 索 请 求）测试三:教育的负面作用是什么(h2 == '第一节 教育的功能')===============")
# 测试
hybrid_search_by_conditions("教育的负面作用是什么", ranker, filter_query="h2 == '第一节 教育的功能'")



['my_collection_1']
{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}
{'upsert_count': 2, 'ids': [1, 88]}
{'delete_count': 1}
=============== 稠 密 检 索 测试一:教育的负面作用是什么===============
{'id': 15, 'distance': 0.6299864053726196, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}
=============== 稠 密 检 索 测试二:给教师的建议是谁写的===============
{'id': 1, 'distance': 0.49007099866867065, 'entity': {'content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。', 'h2': '测试一下'}}
{'id': 6, 'distance': 0.45415082573890686, 'entity': {'content': '### （六）希腊三贤  \n**苏格拉底**：产婆术（谈话法），分三步：讽刺—定义—助产术。国外启发式教育第一人。  \n**柏拉图**：《理想国》提出普及教育的主张，教育目的是培养哲学王。  \n**亚里士多德**：提出自由教育，提倡对学生进行和谐全面发展的教育，灵魂说。', 'h2': '第一节 中外教